# Training PCE Dataset

This notebook generates input samples, evaluates the trained PCE metamodels over time, organizes the results in a DataFrame, and saves the training dataset as CSV.

## 1. Libraries

In [1]:
from pathlib import Path

import dill
import numpy as np
import pandas as pd
from UQpy.distributions import Uniform, JointIndependent

c:\git-projetos\2024-1_victor_hugo_renata_maria\.venv\Lib\site-packages\UQpy\__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## 2. Paths

The PCE metamodels are stored in `beam_problem_1/01_glam_real_data`. The CSV is saved in the project root.

In [2]:
cwd = Path.cwd()
candidate_base_dir = cwd / "beam_problem_1" / "01_glam_real_data"

if candidate_base_dir.exists():
    project_root = cwd
    base_dir = candidate_base_dir
else:
    base_dir = cwd
    project_root = base_dir.parents[1]

output_file = project_root / "training_pce_dataset.csv"

## 3. Input Distributions

The random variables are concrete compressive strength (`fck_dist`) in MPa, relative humidity (`rh_dist`) in %, and concrete cover (`cov_dist`) in mm.

In [3]:
fck_min = 20  # MPa
fck_max = 50  # MPa
rh_min  = 20  # %
rh_max  = 80  # %
cov_min = 15  # mm
cov_max = 60  # mm

installation_year   = "1990"
cement_type         = 3
exposure_conditions = "2"

In [4]:
fck_dist = Uniform(loc=fck_min, scale=fck_max - fck_min)
rh_dist  = Uniform(loc=rh_min, scale=rh_max - rh_min)
cov_dist = Uniform(loc=cov_min, scale=cov_max - cov_min)
joint    = JointIndependent(marginals=[fck_dist, rh_dist, cov_dist])

## 4. Generate Samples

Each sample contains `[fck_mpa, rh_percent, cover_mm]`.

In [5]:
n_samples = 50000
x_pce_rvs = joint.rvs(n_samples)

## 4.1 summary

In [6]:
print("Samples generated successfully!")
print(f"   Number of design samples: {n_samples}")
print("\nSample statistics:")
print(f"   fck:      {x_pce_rvs[:, 0].min():.1f} - {x_pce_rvs[:, 0].max():.1f} MPa (mean: {x_pce_rvs[:, 0].mean():.1f} MPa)")
print(f"   RH:       {x_pce_rvs[:, 1].min():.1f} - {x_pce_rvs[:, 1].max():.1f}% (mean: {x_pce_rvs[:, 1].mean():.1f}%)")
print(f"   cover:    {x_pce_rvs[:, 2].min():.1f} - {x_pce_rvs[:, 2].max():.1f} mm (mean: {x_pce_rvs[:, 2].mean():.1f} mm)")

Samples generated successfully!
   Number of design samples: 50000

Sample statistics:
   fck:      20.0 - 50.0 MPa (mean: 35.0 MPa)
   RH:       20.0 - 80.0% (mean: 50.1%)
   cover:    15.0 - 60.0 mm (mean: 37.5 mm)


## 5. PCE Metamodel Files

In [7]:
times = np.linspace(0, 150, 10, endpoint=True)

pce_files = [
                "100000_pce_metamodel_0.0_install_1990_cement_3_exposure_2.pkl",
                "100000_pce_metamodel_16.666666666666668_install_1990_cement_3_exposure_2.pkl",
                "100000_pce_metamodel_33.333333333333336_install_1990_cement_3_exposure_2.pkl",
                "100000_pce_metamodel_50.0_install_1990_cement_3_exposure_2.pkl",
                "100000_pce_metamodel_66.66666666666667_install_1990_cement_3_exposure_2.pkl",
                "100000_pce_metamodel_83.33333333333334_install_1990_cement_3_exposure_2.pkl",
                "100000_pce_metamodel_100.0_install_1990_cement_3_exposure_2.pkl",
                "100000_pce_metamodel_116.66666666666667_install_1990_cement_3_exposure_2.pkl",
                "100000_pce_metamodel_133.33333333333334_install_1990_cement_3_exposure_2.pkl",
                "100000_pce_metamodel_150.0_install_1990_cement_3_exposure_2.pkl",
            ]

## 6. Build Training Dataset

For each time value, the corresponding PCE metamodel predicts four lambda responses. The final dataset columns are:

- `fck_mpa`: concrete compressive strength in MPa
- `rh_percent`: relative humidity in %
- `cover_mm`: concrete cover in mm
- `tempo`: time associated with each PCE metamodel
- `lambda_1` to `lambda_4`: lambda responses predicted by the PCE metamodels

In [8]:
dataset_rows = []

for j, t in enumerate(times):
    filename = base_dir / pce_files[j]

    with open(filename, "rb") as f:
        pce_metamodel_new = dill.load(f)

    y_pce_new_dataset_pred = pce_metamodel_new.predict(x_pce_rvs)
    time_column = np.full((n_samples, 1), t)
    dataset_rows.append(np.column_stack((x_pce_rvs, time_column, y_pce_new_dataset_pred)))

dataset = np.vstack(dataset_rows)

df = pd.DataFrame(
                        dataset,
                        columns=[
                                    "fck_mpa",
                                    "rh_percent",
                                    "cover_mm",
                                    "tempo",
                                    "lambda_1",
                                    "lambda_2",
                                    "lambda_3",
                                    "lambda_4",
                                ],
                    )

## 7. Save CSV

In [9]:
df.to_csv(output_file, index=False)

## 8. Preview

In [10]:
df.head()

,fck_mpa,rh_percent,cover_mm,tempo,lambda_1,lambda_2,lambda_3,lambda_4
0,45.112461,68.495783,58.585939,0.0,58.371413,1.205343,0.146446,0.123163
1,29.548132,46.088905,48.047686,0.0,47.771502,1.524416,0.146890,0.124164
2,23.365761,37.475105,59.122581,0.0,58.524638,1.170393,0.145583,0.122398
3,38.902050,62.777637,30.773704,0.0,30.795922,2.371339,0.146712,0.123206
4,21.864117,41.705705,21.873372,0.0,20.813607,3.289381,0.144984,0.124602
